In [1]:
datasets = ["sine_rw10_mode5", "weather"]
analyses = ["prequential"]
metrics = ["kappa"]

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import seaborn as sns

In [ ]:
path_read = "/Users/Sandro/OneDrive/Documents/Uni/TESI/GIN/performance/TESTS_8concept"
path_write = "/Users/Sandro/OneDrive/Documents/Uni/TESI/GIN/graphs"

In [ ]:
def rename_models(x):
    if x=="F":
        x="M"
    rename_dict = {
        "D": "DYNcPNN",
        "cG": "cGRU",
        "cP": "cPNN",
        "A": "ARF$_T$",
        "GIN":"GIN"
    }
    if x in rename_dict:
        return(rename_dict[x])
    else:
        return x
    
rename_dataset = {
    "sine_rw10_mode5": "SRWM",
    "weather": "Weather"
}
    

In [5]:
import scienceplots

In [6]:
import numpy as np

In [7]:
df = pd.DataFrame()
df["y"] = np.random.uniform(0, 1, 100)
df["x"] = np.arange(0,100)

In [ ]:
from matplotlib.lines import Line2D

cm = 1 / 2.54

def plot(analysis, metric, dataset):
    df = pd.read_csv(os.path.join(path_read, f"{analysis}_concept_{metric}_{dataset}.csv"))
    #df = df[df["model"]!="A"]
    df["model_lower"] = df["model"].apply(lambda x : x.lower())
    df = df.sort_values("model_lower").drop(columns="model_lower")
    df["style"] = "solid"
    df["model"] = df["model"].apply(rename_models)
    models = list(df["model"].unique())

    plt.style.use(['science', 'ieee'])
    plt.rcParams.update({'font.size': 9})

    plot = sns.lineplot(
        data=df,
        x="timestamp", y=metric, errorbar="sd", hue="model", style="style",
        #palette={"A": "C3", "cPNN": "C0", "cLSTM": "C1", "DYNcPNN": "C2"},
        palette={"ARF$_T$": "#9467bd", "cPNN": "#2ca02c", "cGRU": "#1f77b4", "GIN": "#ff7f0e"}
    )

    lines = plot.get_lines()
    colors = [lines[i].get_color() for i in range(len(lines))][:len(models)]

    plt.legend().remove()
    handles, labels = plt.gca().get_legend_handles_labels()
    lines = [Line2D([0], [0], label=m, color=c, linestyle="-") for c, m in zip(colors, models)]
    plt.legend(handles=lines)
    plt.xlabel("N. of data points within the concept", loc="right")
    if metric=="accuracy":
        plt.ylabel("Balanced Accuracy", loc="top")
    else:
        plt.ylabel("Cohen's Kappa", loc="top")

    n = len(df[(df["model"]==models[1]) & (df["conf"]==1)])
    plt.xlim(0, n)
    y_min = df[df["timestamp"]>2000][metric].min()
    y_max = df[df["timestamp"]>2000][metric].max()
    plt.ylim(y_min, y_max)
    plt.axvline(x=50*128, color="grey", linestyle="dashed", linewidth=1)
    plt.text(50*128, y_min+0.01, f'start', rotation=90, va='bottom', ha='left', color="grey")
    plt.text(n, y_min+0.01, f'end', rotation=90, va='bottom', ha='left', color="grey")

    fig = plt.gcf()
    fig.set_size_inches(7*cm, 6.5*cm)
    plt.title(rename_dataset[dataset])
    plt.minorticks_off()
    plt.tight_layout()
    plt.savefig(
        f"{path_write}/performance_concept_{analysis}_{metric}_{dataset}.png",
        transparent=True,
        dpi = 1000
    )
    plt.close(fig)

In [ ]:
for analysis in analyses:
    for metric in metrics:
        for dataset in datasets:
            print(analysis, metric, dataset)
            plot(analysis, metric, dataset)

prequential kappa sine_rw10_mode5
